# System-design drills & self-scoring rubrics

> 📓 *Companion to* **Modern Agentic AI Engineer** · Ch 52 §52.1–52.3 · type: worksheet

**The promise:** by the end you will have run the chapter's five-phase AI system-design framework *cold* on three fresh agent-product prompts, scored each against a rubric that rewards spoken trade-offs and *unprompted* evaluate-and-operate, drafted a leveled behavioral story-bank, and filled a leveling + negotiation sheet you keep.

This is a *worksheet*, not a code lab. It runs fully offline and free — no API key, no network, just the standard library and a little string assembly. You edit the ✍️ fill-in cells; a few tiny cells score what you wrote and tell you where you're weak. **The interview is the book, compressed** (§52.1) — this worksheet is the rehearsal.

## 🧠 Why this matters

You are not walking into the room to pass a quiz; you are there to give the interviewer *evidence they can advocate with* in the debrief (§52 intro). The system-design round is where senior-and-above offers are won, and for agentic roles it is increasingly *the* differentiating round — because most candidates narrate an architecture and stop, while the rare ones reach **evaluation and operations unprompted**.

Reading the five-phase framework and *performing* it aloud under time pressure are different skills, and only the second one transfers to the room. The field is new enough that rubrics are uneven and interviewers are improvising — a candidate who brings their own structure steers the conversation. That structure is the framework you already built across this whole book; this worksheet makes running it a reflex instead of a recital.

## Objectives & prereqs

**By the end you can:**
- Run the five phases (Clarify → Rank the -ilities → Shape → Deep-dive → Evaluate & operate) cold on an unseen agent-product prompt.
- Reach **evaluate-and-operate unprompted** — the rarest, highest-signal move.
- Self-score a design answer against a rubric that grades *spoken* trade-offs, not silent correctness.
- Tell six to eight behavioral stories at the highest level that is *true*, with your specific actions and measured results.
- Walk into leveling + negotiation with a researched plan and at least one alternative process in hand.

**Prereqs:** Ch 52 read. **Ch 42** (the system-design method these drills perform) — ideally after running [`42-02-run-the-method-worksheet.ipynb`](../../part-11-architecting-at-scale/42-system-design-for-ai/42-02-run-the-method-worksheet.ipynb). **Ch 27** (the -ilities you rank and sacrifice in Phase 2). **Ch 50** (the level ladder your stories map onto). The coding-round prep is *deliberately not here* — it lives in Part II's runnable notebooks; this worksheet drills the rounds code can't rehearse.

**Packages:** standard library only (`textwrap`, `dataclasses`). Nothing beyond `requirements.txt`; no API key; fully offline.

In [ ]:
# Setup — imports, env, and the MOCK switch.
import os
import textwrap
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# This worksheet is pure local reflection + string assembly — it makes NO model
# calls in either mode, and (per §52.3) hardcodes NO market-comp numbers, since
# any book's numbers go stale. We still read COMPANION_MOCK so the habit and the
# message stay consistent across the whole course (see docs/NOTEBOOK-STANDARDS.md).
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

print("Interview drill worksheet — offline, free, no API key needed.")
print(f"COMPANION_MOCK = {os.getenv('COMPANION_MOCK', '1')} "
      "(this notebook makes no model calls either way)")
print()
print("The worked-example lines come from §52.1's customer-support walkthrough.")
print("Replace each ✍️ cell with YOUR answers; the worked moves are there to imitate.")

## 🧠 The five phases — the book's method, performed aloud

The framework is nothing more than the book's design method (Ch 42) spoken under time pressure. Run it *in order*; you cannot honestly deep-dive (Phase 4) before you've ranked the -ilities (Phase 2).

| # | Phase | What you do | Signal it sends |
|---|---|---|---|
| 1 | **Clarify** | Interrogate users, scale, latency/error tolerance, what "good" means — and pin **act vs. only say** (the autonomy budget). | You don't build before understanding. |
| 2 | **Rank the -ilities** | Name the 2–3 quality attributes that dominate (Ch 27) and say what you'll **sacrifice**. | You think in trade-offs, not features. |
| 3 | **Shape** | Boxes and boundaries (model, orchestration, tools, retrieval, state, human-in-the-loop); narrate *why* each boundary falls where it does. | Architectural judgment, live. |
| 4 | **Deep-dive the hard parts** | Pick the riskiest components: failure modes, guardrails, cost controls, prompt injection, context management. | Depth on demand; you know where the danger lives. |
| 5 | **Evaluate & operate** | How you'd measure (evals, baselines), observe (traces), and run it (cost, latency, incidents, staged rollout). | You ship systems, not demos — the rarest signal. |

> 🎯 **Phase 5 is the differentiator.** Most candidates never get past architecture into evaluation and operations *unprompted*. This worksheet weights it accordingly — reaching it on your own is worth more than a prettier diagram.

### Study the worked example first (§52.1)

Prompt: *"Design an AI assistant that handles our customer-support tickets."* Read how a senior runs all five phases on it — this is the shape you're about to reproduce on fresh prompts.

In [ ]:
WORKED_EXAMPLE = {
    "prompt": "Design an AI assistant that handles our customer-support tickets.",
    "clarify": (
        "Fraction of simple vs. account-action tickets? Drafting for human review or "
        "replying autonomously? Cost of a wrong answer — annoyance, or money + legal? "
        "=> autonomy budget. Assume: high volume, acts autonomously on low-risk "
        "intents, escalates the rest."
    ),
    "rank": (
        "Correctness + safety dominate; latency secondary (support is async-tolerant); "
        "cost matters at volume. SACRIFICE: trade some resolution rate for a low "
        "false-action rate."
    ),
    "shape": (
        "Cheap intent classifier triages. Low-risk info tickets -> RAG-grounded "
        "responder over the KB. Account-action tickets -> tool-using agent with "
        "SCOPED, REVERSIBLE tools (refund up to a limit, never delete). Above a risk "
        "threshold -> human queue. State lives outside the agent; every action logged "
        "with the full trace."
    ),
    "deep_dive": (
        "Tool layer: idempotent, permission-scoped, hard caps. Prompt injection via "
        "ticket content is a FIRST-CLASS threat (a hostile email is untrusted input). "
        "Escalation boundary: estimate confidence, tune from production data not guesses."
    ),
    "evaluate_operate": (
        "Eval set from historical tickets with known-good resolutions; measure "
        "resolution rate, false-action rate, escalation precision. Ship shadow -> "
        "assist -> bounded autonomy, each gate passed on eval numbers not vibes. "
        "Dashboards on cost/ticket and drift."
    ),
}

for phase, text in WORKED_EXAMPLE.items():
    if phase == "prompt":
        print(f"PROMPT: {WORKED_EXAMPLE['prompt']}\n" + "=" * 72)
        continue
    print(f"\n[{phase.replace('_', ' & ').upper()}]")
    print(textwrap.fill(text, 76))

## ✍️ Drill cards — run all five phases on three fresh prompts

Here are three prompts the chapter doesn't work through. **Pick one and fill all five phases before reading the next** — the point is to run the framework *cold*, the way the room will demand it. Then do the other two. Keep each phase to a few spoken sentences; this is a script you'd say out loud, not an essay.

1. **Meeting-notes agent** — joins internal calls, produces summaries + action items, can create tickets/calendar invites.
2. **Code-review bot** — comments on pull requests, can suggest diffs, optionally approve low-risk changes.
3. **Research assistant over internal docs** — answers employee questions across wikis, contracts, and tickets with citations.

Fill in the blank strings below. The worked example above is your model for the shape and depth of each phase.

In [ ]:
@dataclass
class Drill:
    prompt: str
    clarify: str = ""          # Phase 1: incl. ACT vs ONLY-SAY (autonomy budget)
    rank: str = ""             # Phase 2: top 2-3 -ilities + what you SACRIFICE
    shape: str = ""            # Phase 3: boxes + boundaries, with WHY each one falls there
    deep_dive: str = ""        # Phase 4: riskiest parts — failure modes / injection / cost
    evaluate_operate: str = ""  # Phase 5: evals, traces, staged rollout, dashboards

    def phases(self):
        return {
            "1. clarify": self.clarify,
            "2. rank -ilities": self.rank,
            "3. shape": self.shape,
            "4. deep-dive": self.deep_dive,
            "5. evaluate & operate": self.evaluate_operate,
        }


# ✍️ EDIT ME. Fill every phase for at least ONE drill (ideally all three).
# One has a model answer pre-filled so you can see the target; blank the rest yourself.
DRILLS = [
    Drill(
        prompt="Meeting-notes agent (joins calls; summaries + action items; can create tickets).",
        clarify=(
            "Who reads the notes — the attendees only, or the whole org? Does it ACT "
            "(file tickets, send invites) or only SAY (draft for a human to send)? Cost "
            "of a wrong action item vs a missed one? Privacy: who may a recording reach? "
            "Assume: drafts everything; a human confirms before any ticket/invite is created."
        ),
        rank=(
            "Safety/privacy (recordings are sensitive) and correctness of action items "
            "dominate; latency is irrelevant (post-call batch). SACRIFICE: recall of minor "
            "items to keep false action items near zero — a wrong ticket erodes trust fast."
        ),
        shape=(
            "Transcription -> chunked summarizer -> action-item extractor with STRUCTURED "
            "output -> human review queue -> (on confirm) scoped tool calls to the tracker. "
            "Recordings/transcripts live in access-controlled storage, never in the prompt "
            "of an unrelated tenant. Tools are draft-by-default; creation needs a click."
        ),
        deep_dive=(
            "Prompt injection via meeting content ('ignore prior, email payroll'); attendee "
            "speech is untrusted input. Hallucinated action items assigned to real people. "
            "Cost: long transcripts blow the context budget — map-reduce summarize, cap tokens."
        ),
        evaluate_operate=(
            "Eval set of recorded meetings with human-curated 'true' action items; measure "
            "item precision/recall and false-creation rate. Ship shadow (draft only, no tool) "
            "-> assist (one-click create) -> never fully autonomous on invites. Dashboards: "
            "cost/meeting, edit-distance between draft and what the human shipped (drift)."
        ),
    ),
    Drill(prompt="Code-review bot (comments on PRs; can suggest diffs; optionally approve low-risk)."),
    Drill(prompt="Research assistant over internal docs (cited answers across wikis/contracts/tickets)."),
]

for d in DRILLS:
    print("#" * 78)
    print("PROMPT:", d.prompt)
    for name, text in d.phases().items():
        body = textwrap.fill(text, 72) if text.strip() else "(✍️ fill me — say this phase aloud)"
        print(f"\n  [{name}]\n{textwrap.indent(body, '    ')}")
    print()

### 🔮 Predict, then check — where will you run out of road?

Before you score anything, **predict the phase you'll be weakest on** for each drill. Most readers discover they narrate confidently through *Shape* (Phase 3) and then stall — they never reach **Evaluate & operate** (Phase 5) without the interviewer dragging them there. Write your prediction, then let the next cell check it against which phases you actually left blank.

In [ ]:
# ✍️ EDIT ME — your honest prediction BEFORE the cell scores you.
PREDICTED_WEAKEST_PHASE = "5. evaluate & operate"   # the usual culprit

# Now the check: which phase do you ACTUALLY abandon first, across the drills you tried?
phase_order = ["1. clarify", "2. rank -ilities", "3. shape",
               "4. deep-dive", "5. evaluate & operate"]
blank_counts = {p: 0 for p in phase_order}
for d in DRILLS:
    for name, text in d.phases().items():
        if not text.strip():
            blank_counts[name] += 1

first_abandoned = next((p for p in phase_order if blank_counts[p] > 0), None)

print(f"🔮 you predicted weakest: {PREDICTED_WEAKEST_PHASE}")
print("blanks per phase:")
for p in phase_order:
    bar = "█" * blank_counts[p]
    print(f"  {p:24s} {blank_counts[p]}  {bar}")
print()
if first_abandoned is None:
    print("✓ every phase filled on every drill — now do it ALOUD, on a timer, cold.")
elif first_abandoned == PREDICTED_WEAKEST_PHASE:
    print(f"✓ you predicted it: '{first_abandoned}' is where you stop. That's the muscle to train.")
else:
    print(f"⚠️ surprise: you actually stop earliest at '{first_abandoned}', "
          f"not the '{PREDICTED_WEAKEST_PHASE}' you predicted. Re-train that phase first.")

## 📋 Self-scoring rubric — grade the *spoken reasoning*, not the answer

Score each drill honestly. The rubric rewards exactly the senior signals the chapter names, and it weights **Phase 5 (evaluate & operate, unprompted)** highest — because that's the move that separates you from the field. Score each row 0–2:

- **0** = didn't do it · **1** = did it but silently / shallowly · **2** = did it and *narrated the why* aloud.

Note the grain: a "wrong" architecture choice you *narrated a trade-off for* scores higher than a "right" one you made silently. The interviewer grades what they can hear.

In [ ]:
@dataclass
class RubricRow:
    criterion: str
    weight: int        # Phase 5 weighted highest — it's the differentiator
    score: int = 0     # ✍️ you fill this: 0 / 1 / 2


def fresh_rubric():
    # Mirrors the §52 checklist + the five-phase signals; Phase 5 carries 2x weight.
    return [
        RubricRow("Clarified AUTONOMY ('act' vs 'only say') before designing", 1),
        RubricRow("Ranked the -ilities AND named what you'd SACRIFICE", 1),
        RubricRow("Narrated WHY each boundary falls where it does (not just boxes)", 1),
        RubricRow("Deep-dived FAILURE modes / prompt injection / COST — not happy paths", 1),
        RubricRow("Reached EVALS (set + baselines + metrics) UNPROMPTED", 2),
        RubricRow("Reached staged ROLLOUT (shadow->assist->bounded) UNPROMPTED", 2),
        RubricRow("Reached OPS (cost/drift dashboards, traces) UNPROMPTED", 2),
    ]


# ✍️ EDIT ME — score the FIRST drill (the worked meeting-notes answer) as practice,
# then copy this cell and re-score for the drills you filled yourself.
DRILL_UNDER_REVIEW = DRILLS[0].prompt
rubric = fresh_rubric()
scores = [2, 2, 2, 2, 2, 1, 1]   # <- replace each with YOUR honest 0/1/2
for row, s in zip(rubric, scores):
    row.score = s

earned = sum(r.score for r in rubric)
possible = sum(2 for _ in rubric)
phase5 = [r for r in rubric if r.weight == 2]
phase5_earned = sum(r.score for r in phase5)
phase5_possible = sum(2 for _ in phase5)

print(f"Scoring: {DRILL_UNDER_REVIEW}\n" + "-" * 72)
for r in rubric:
    star = " ★(differentiator)" if r.weight == 2 else ""
    print(f"  [{r.score}/2] {r.criterion}{star}")
print("-" * 72)
print(f"  total: {earned}/{possible}   |   PHASE 5 (unprompted eval+ops): "
      f"{phase5_earned}/{phase5_possible}")
print()
if phase5_earned < phase5_possible:
    print("⚠️ You left Phase-5 points on the table. In a real loop, reaching evals,")
    print("   staged rollout, and ops dashboards WITHOUT being asked is the whole game.")
else:
    print("✓ You reached evaluate-and-operate unprompted — that's the senior signal.")

### ⚠️ Pitfall: making choices the interviewer silently disagrees with

The most expensive thing you can do in a design round is make a defensible choice *silently*. Interviewers can't read your mind, and most candidates quietly pick an option the interviewer quietly disagrees with — the disagreement is never aired, and the candidate loses points they could have won. **The fix is the rubric you just used**: it scores *spoken* trade-off reasoning, which converts even a "wrong" choice into senior signal.

The script that does this: state the choice, the alternative, and your trip-wire. *"I'd start with a single orchestrator rather than multi-agent — coordination overhead isn't justified at this scale; here's what would change my mind."* The cell below is a tiny template for narrating a trade-off out loud.

In [ ]:
def narrate_tradeoff(choice, alternative, why, what_would_change_my_mind):
    return textwrap.fill(
        f"I'd go with {choice} rather than {alternative}, because {why}. "
        f"What would change my mind: {what_would_change_my_mind}.", 76)


# ✍️ EDIT ME — narrate one real choice from a drill above, out loud, then read it back.
print(narrate_tradeoff(
    choice="a single orchestrator",
    alternative="a multi-agent topology",
    why="coordination overhead isn't justified at this scale and it's easier to eval",
    what_would_change_my_mind="clearly separable skills with independent failure domains",
))
print()
print("Rule of thumb: if you changed the design in your head and DIDN'T say why,")
print("you just donated a point to the committee's doubts. Say the why every time.")

## ✍️ Behavioral story-bank — six to eight true stories, leveled

Behavioral rounds are **leveling rounds in disguise** (§52.2): interviewers map your stories onto the Ch 50 ladder — did you *execute a task*, *own a system*, or *align teams*? Draft six to eight **true** stories, each as situation / **your specific actions** / measured result, told at the highest level that is true. Cover the canonical set: a hard technical call, a conflict, a failure and what it changed in you, and a time you multiplied someone else.

In [ ]:
@dataclass
class Story:
    label: str
    situation: str = ""
    your_actions: str = ""   # YOUR part, specifically — not the team's
    measured_result: str = ""  # a NUMBER where one exists
    level_signal: str = ""   # execute task | own system | align teams (Ch 50)


# ✍️ EDIT ME — aim for 6–8. One is filled as a model; replace ALL with your truth.
STORY_BANK = [
    Story(
        label="Hard technical call",
        situation="Our agent's tool layer kept double-charging on retries under load.",
        your_actions=(
            "I wrote the RFC for idempotency keys per ticket-step, aligned three teams on "
            "the boundary, and personally built the eval gate that blocked the regression."
        ),
        measured_result="Double-charges went from ~12/week to 0 over two release cycles.",
        level_signal="align teams",
    ),
    Story(label="Conflict / disagreement"),
    Story(label="A failure + what it changed in me"),
    Story(label="A time I multiplied someone else"),
    Story(label="Ambiguous problem I scoped"),
    Story(label="A bet that paid off (or didn't)"),
]

filled = [s for s in STORY_BANK if s.situation.strip() and s.your_actions.strip()]
print(f"Story-bank: {len(filled)}/{len(STORY_BANK)} drafted (target 6–8)\n" + "=" * 72)
for s in STORY_BANK:
    state = "✓" if s in filled else "✍️"
    print(f"{state} [{s.level_signal or '?':>11s}] {s.label}")
    if s in filled:
        print(textwrap.indent(textwrap.fill(
            f"S: {s.situation}  A: {s.your_actions}  R: {s.measured_result}", 72), "     "))
if len(filled) < 6:
    print(f"\n⚠️ Draft {6 - len(filled)} more — you want range, so any panel finds the right fit.")

### ⚠️ Pitfall: "we" is a level-killer

*"We migrated the platform"* gives the interviewer nothing to grade **you** on, and ambiguity about your role is always resolved **downward**. Precision about your part isn't arrogance — vagueness is just a donation to the committee's doubts. Take one story and rewrite the vague "we" version into your precise part: *"I wrote the RFC, aligned three teams on the boundary, and built the eval gate."*

In [ ]:
# ✍️ EDIT ME — rewrite a 'we' sentence into YOUR specific actions.
WE_VERSION = "We migrated the platform to the new agent runtime and cut costs."
I_VERSION = (
    "I wrote the migration RFC, aligned the three owning teams on the cutover "
    "boundary, and built the eval gate that let us ship without a quality regression "
    "— which is what took cost/ticket from $0.31 to $0.018."
)

import re
we_count = len(re.findall(r"\bwe\b|\bour\b", WE_VERSION, flags=re.I))
i_count = len(re.findall(r"\bI\b|\bmy\b", I_VERSION))
print("BEFORE (level-killer):", WE_VERSION)
print(f"   -> {we_count} 'we/our', {len(re.findall(chr(73), WE_VERSION))} 'I' — nothing to grade you on.\n")
print("AFTER  (gradeable):")
print(textwrap.fill(I_VERSION, 76))
print(f"   -> {i_count} first-person + a number. THIS is what the committee can level.")

## ✍️ Leveling & negotiation sheet

Two decisions are made about you after the loop: **level** and **offer** (§52.3). Most candidates negotiate the offer and ignore the level — backwards, because level sets your scope, your peers, and the baseline every future raise compounds from. Fill the sheet below. Note what it deliberately leaves **blank**: the market range, which *you* research on current data for *this* company — the chapter is explicit that any book's comp numbers go stale, so this notebook hardcodes none.

In [ ]:
# ✍️ EDIT ME. Research the market range yourself — do NOT trust any stale number.
LEVELING_SHEET = {
    "target_level": "",          # e.g. 'Senior (L5-equivalent)'
    "that_level's_scope": "",    # what someone at this level OWNS (Ch 50 ladder)
    "researched_range_for_THIS_company": "",  # YOU fill from current market data
    "my_anchor_if_forced": "",   # top of YOUR researched range, not a guess
    "down_level_trap_questions_in_writing": (
        "If offered one level down with 'fast promotion': get the timeline AND criteria "
        "in writing. What exactly must be true to promote, and by when?"
    ),
    "parallel_process_to_create_an_alternative": "",  # a competing offer moves numbers
    "leveling_evidence_if_under-leveled": (
        "Portfolio system from Ch 50 / capstone/, a design doc, references — ask what "
        "signal was missing and whether more evidence can be considered."
    ),
}

print("Leveling & negotiation sheet\n" + "=" * 60)
blanks = []
for k, v in LEVELING_SHEET.items():
    shown = v if v.strip() else "(✍️ fill / research me)"
    if not v.strip():
        blanks.append(k)
    print(f"• {k.replace('_', ' ')}:")
    print(textwrap.indent(textwrap.fill(shown, 70), "    "))

print("\n" + "-" * 60)
if blanks:
    print("⚠️ Still blank (these are the ones that win or lose the negotiation):")
    for b in blanks:
        print(f"   - {b.replace('_', ' ')}")
print("\nThe durable principles (§52.3): leverage is ALTERNATIVES; let them anchor;")
print("negotiate the PACKAGE not just salary; be unfailingly warm + willing to walk.")

## 🎯 Senior lens

Every cell in this worksheet serves one purpose: **generating evidence the interviewer can advocate with** in the debrief — not passing a quiz. That reframe changes how you run each phase. You narrate trade-offs not to sound smart but because a spoken trade-off is a *gradeable artifact* the interviewer carries into the room where your level is decided. You reach evaluate-and-operate unprompted not to check a box but because it's the one signal that says *I have shipped this before*, and it's the rarest thing on the panel's scorecard.

So the real test of this worksheet isn't whether the cells are filled — it's whether, by the end, you could run all five phases *aloud, on a timer, on a prompt you've never seen*, and reach Phase 5 without being dragged there. The market doesn't pay you what you're worth; it pays what the evidence and the alternatives in front of it justify. This worksheet is where you manufacture both.

## Recap

- The interview prices years of capability in a few hours — you're there to hand over **evidence to advocate with**, not to pass a quiz.
- Run the **five phases** (Clarify → Rank the -ilities → Shape → Deep-dive → Evaluate & operate) cold; **Phase 5, unprompted, is the differentiator**.
- Clarify the **autonomy budget** (act vs. only say) and **rank + sacrifice** the -ilities before drawing any boxes.
- The rubric grades **spoken** trade-offs — a narrated "wrong" choice beats a silent "right" one.
- Behavioral rounds are **leveling rounds**: 6–8 true stories, *your* specific actions, measured results; "we" is a **level-killer**.
- Negotiate the **level** first, then the package; your leverage is **alternatives**, and you research current comp yourself — no book's numbers survive.

## Exercises

Each exercise *changes* something and asks you to **predict** the result before you run it.

1. **Drill cold, on a timer.** Pick the code-review-bot or research-assistant drill you left blank, set a 15-minute timer, and fill all five phases *out loud* before writing. Predict which phase you'll rush; then score it with a fresh rubric. Did you reach Phase 5 before the timer?
2. **Re-weight the rubric.** Change the three Phase-5 rows to weight 1 (flatten the rubric). Predict how a candidate who stops at *Shape* now scores — does the rubric still distinguish senior from mid? Argue why Phase 5 must stay 2x.
3. **Level a story up and down.** Take one story and write two versions: the "execute a task" framing and the truthful "align teams" framing. Predict which the panel grades higher, and confirm the higher one is still *true* (if it isn't, you found your real level).
4. **Run the down-level trap.** Draft the exact written questions you'd send if an offer came one level low with a "fast-promotion" promise. Predict which answer would make you walk.

In [ ]:
# Exercise scratch space — your code/notes here.


In [ ]:
# Exercise scratch space — your code/notes here.


## Next

- **Template:** the drills practice the same scaffold as [`templates/system-design-doc/`](../../../templates/system-design-doc/) (C4 + -ilities + trade-offs) — interview reps and on-the-job design docs are one muscle. Run [`42-02-run-the-method-worksheet.ipynb`](../../part-11-architecting-at-scale/42-system-design-for-ai/42-02-run-the-method-worksheet.ipynb) to produce a full worked instance of that template.
- **Book:** §52.1 (the five-phase framework + customer-support worked example), §52.2 (coding/ML/behavioral rounds), §52.3 (negotiation & leveling), closing on the §52 checklist. The coding-round prep lives in **Part II**'s runnable notebooks — drill those separately.
- **Capstone:** there's no code here, but the portfolio system from `capstone/` (Appendix C) is your **leveling evidence** when an offer lands below the level you interviewed for — §52.3 names it explicitly.